In [ ]:
from datetime import datetime

job_id = session.sql("SELECT UUID_STRING()").collect()[0][0]
job_name = "DWH_FACT_CASES_LOAD"
layer_name = "DWH"
start_time = datetime.now()

rows_loaded = 0
rows_failed = 0
run_status = "SUCCESS"
error_message = None

try:

    fact_cases_df = session.sql(f"""
        SELECT
            c.CAS_ID,
            c.CASE_NAME,
            d.CASES_INFO_ID,
            c.RESTRICTED_IND,
            c.CASE_ACCESS_IND,
            TRY_TO_NUMBER(c.ASSIST_CASE_NUM) AS ASSIST_CAS_NUM,
            c.EXPECTED_CLOSE_DATE,
            c.UNIT_ORGN_ID,
            c.AREA_ORGN_ID,
            c.REGION_ORGN_ID,
            c.CURRENT_CASE_STATUS_DATE,
            c.RESTRICTION_DATE,
            c.ADD_TS,
            c.ADD_USER,
            c.MOD_TS,
            c.MOD_USER,
            CURRENT_TIMESTAMP() AS CREATED_DATE,
            c.DELETED_DATE,
            MD5(
                COALESCE(c.CASE_NAME, '') || '|' ||
                COALESCE(TO_VARCHAR(d.CASES_INFO_ID), '') || '|' ||
                COALESCE(c.RESTRICTED_IND, '') || '|' ||
                COALESCE(c.CASE_ACCESS_IND, '') || '|' ||
                COALESCE(TO_VARCHAR(TRY_TO_NUMBER(c.ASSIST_CASE_NUM)), '') || '|' ||
                COALESCE(TO_VARCHAR(c.EXPECTED_CLOSE_DATE), '') || '|' ||
                COALESCE(TO_VARCHAR(c.UNIT_ORGN_ID), '') || '|' ||
                COALESCE(TO_VARCHAR(c.AREA_ORGN_ID), '') || '|' ||
                COALESCE(TO_VARCHAR(c.REGION_ORGN_ID), '') || '|' ||
                COALESCE(TO_VARCHAR(c.CURRENT_CASE_STATUS_DATE), '') || '|' ||
                COALESCE(TO_VARCHAR(c.RESTRICTION_DATE), '') || '|' ||
                COALESCE(TO_VARCHAR(c.ADD_TS), '') || '|' ||
                COALESCE(c.ADD_USER, '') || '|' ||
                COALESCE(TO_VARCHAR(c.MOD_TS), '') || '|' ||
                COALESCE(c.MOD_USER, '') || '|' ||
                COALESCE(TO_VARCHAR(c.DELETED_DATE), '')
            ) AS CHECKSUM
        FROM {STG}.{STG_CASES} c
        INNER JOIN {DWH}.{DIM_CASES_INFO} d
            ON c.CASE_TYPE IS NOT DISTINCT FROM d.CASE_TYPE_AV_ID
           AND c.CURRENT_CASE_STATUS_CODE IS NOT DISTINCT FROM d.CURRENT_CASE_STATUS_CODE_AV_ID
           AND c.RESTRICTION_REASON_CODE IS NOT DISTINCT FROM d.RESTRICTION_REASON_CODE_AV_ID
           AND d.UPDATED_DATE IS NULL
    """)

    fact_cases_df.create_or_replace_temp_view("TEMP_FACT_CASES")

    merge_result = session.sql(f"""
        MERGE INTO {DWH}.{FACT_CASES} tgt
        USING TEMP_FACT_CASES src
            ON tgt.CAS_ID = src.CAS_ID
        WHEN MATCHED AND (
           src.CHECKSUM<>tgt.CHECKSUM
        ) THEN
            UPDATE SET
                CASE_NAME = src.CASE_NAME,
                CASES_INFO_ID = src.CASES_INFO_ID,
                RESTRICTED_IND = src.RESTRICTED_IND,
                CASE_ACCESS_IND = src.CASE_ACCESS_IND,
                ASSIST_CAS_NUM = src.ASSIST_CAS_NUM,
                EXPECTED_CLOSE_DATE = src.EXPECTED_CLOSE_DATE,
                UNIT_ORGN_ID = src.UNIT_ORGN_ID,
                AREA_ORGN_ID = src.AREA_ORGN_ID,
                REGION_ORGN_ID = src.REGION_ORGN_ID,
                CURRENT_CASE_STATUS_DATE = src.CURRENT_CASE_STATUS_DATE,
                RESTRICTION_DATE = src.RESTRICTION_DATE,
                ADD_TS = src.ADD_TS,
                ADD_USER = src.ADD_USER,
                MOD_TS = src.MOD_TS,
                MOD_USER = src.MOD_USER,
                DELETED_DATE = src.DELETED_DATE,
                CHECKSUM = src.CHECKSUM
        WHEN NOT MATCHED THEN
            INSERT (
                CAS_ID,
                CASE_NAME,
                CASES_INFO_ID,
                RESTRICTED_IND,
                CASE_ACCESS_IND,
                ASSIST_CAS_NUM,
                EXPECTED_CLOSE_DATE,
                UNIT_ORGN_ID,
                AREA_ORGN_ID,
                REGION_ORGN_ID,
                CURRENT_CASE_STATUS_DATE,
                RESTRICTION_DATE,
                ADD_TS,
                ADD_USER,
                MOD_TS,
                MOD_USER,
                CREATED_DATE,
                DELETED_DATE,
                CHECKSUM
            )
            VALUES (
                src.CAS_ID,
                src.CASE_NAME,
                src.CASES_INFO_ID,
                src.RESTRICTED_IND,
                src.CASE_ACCESS_IND,
                src.ASSIST_CAS_NUM,
                src.EXPECTED_CLOSE_DATE,
                src.UNIT_ORGN_ID,
                src.AREA_ORGN_ID,
                src.REGION_ORGN_ID,
                src.CURRENT_CASE_STATUS_DATE,
                src.RESTRICTION_DATE,
                src.ADD_TS,
                src.ADD_USER,
                src.MOD_TS,
                src.MOD_USER,
                src.CREATED_DATE,
                src.DELETED_DATE,
                src.CHECKSUM
            )
    """).collect()

    rows_inserted = merge_result[0][0]
    rows_updated = merge_result[0][1]
    rows_loaded = rows_inserted + rows_updated

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_CASES load completed. Rows Inserted: {rows_inserted}, Rows Updated: {rows_updated}"
    )

    print(f"DWH LOAD SUCCESS | Rows Inserted: {rows_inserted} | Rows Updated: {rows_updated}")

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1
    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_CASES load failed. Error: {error_message}"
    )

    print(f"DWH LOAD FAILED: {error_message}")